# Python Multithreading, Multiprocessing, and Modern Concurrency

* concurrency vs. parallelism
* I/O-bound vs. CPU-bound work
* practical implications of the GIL
* threading
* multiprocessing
* inter-process communication
* common pitfalls
* higher-level concurrency with __`concurrent.futures`__
* __`asyncio`__
* choosing the right tool

The big idea is this:

> use threads when your program spends time waiting, use processes when your program spends time computing, and use higher-level tools when they make the design clearer


## Concurrency vs. Parallelism

These terms are related, but not the same

* **concurrency**
  * managing multiple tasks so they can make progress over overlapping periods of time
  * often improves responsiveness
* **parallelism**
  * executing multiple tasks at the same time on multiple CPU cores
  * often improves throughput

A useful rule of thumb:

* concurrency is about dealing with many tasks
* parallelism is about doing many tasks at the same time
* [Great talk by Rob Pike](https://www.youtube.com/watch?v=oV9rvDllKEg)

Python supports both ideas, but the right tool depends on the kind of work being done


## I/O-Bound vs. CPU-Bound Work

Before choosing a concurrency tool, identify the kind of workload you have

* **I/O-bound**
  * waiting on external systems
  * examples: API calls, file reads, database queries, sockets
* **CPU-bound**
  * spending time doing computation
  * examples: large loops, image processing, simulations, number crunching

This distinction matters because different concurrency tools help with different bottlenecks


In [ ]:
import time

# I/O tasks spend much of their time sleeping, or *waiting* for something
# to occur...
def fake_io_task():
    time.sleep(1)

# CPU tasks spend much of their time computing, rather than waiting
def fake_cpu_task():
    total = 0
    for i in range(5_000_000):
        total += i
    return total

## Practical Implications of the GIL

* in standard CPython, the Global Interpreter Lock (GIL), means that only one thread executes Python bytecode at a time

That leads to an important practical takeaway:

* threads are usually good for I/O-bound work
* threads usually do **not** speed up CPU-bound Python code
* processes can run truly in parallel across cores

This is why threading and multiprocessing are important to understand and distinguish!

For this course, the key point is practical, not internal:

> the GIL limits CPU-bound threading, but does not prevent useful I/O concurrency


## Threading

* the __`threading`__ module provides low-level thread support in Python

* threads share the same memory space, which makes communication easy, but also introduces shared-state risks

* a basic thread lifecycle looks like:
  * create a __`Thread`__
  * call __`start()`__
  * later call __`join()`__ to wait for completion


In [ ]:
# old school...we have to create a Class
# later examples do not require Classes

import threading

# A shared flag to tell the thread when to stop
stop_event = threading.Event()

class Worker(threading.Thread):
    """Create a new class which inherits from Thread"""
    
    def run(self):
        """The run() method is where the Thread starts. We typically would not call the .run()
        method() directly, but instead, once the class is "instantiated" (i.e., an instance of
        the class is created), we call the .start() method.
        """
        print('Worker started')
        count = 1

        # Busy work...perform multiplication repeatedly until the shared flag
        # (stop_event) lets us know we should stop.
        while not stop_event.is_set():
            result = count * count
            count += 1

        # Let us know how many multiplications occurred
        print(f'Worker calculated squares up to {count:,} * {count:,} = {result:,}')

# Create the background thread
thread = Worker()

# In the main thread, wait for user to press ENTER
input("Press ENTER to START the worker...")

# Once the user hits ENTER, the thread is started
thread.start()

# The worker thread is started, and now the main thread is waiting for user input
input("Press Enter to STOP the worker...")

# Signal the worker to stop
stop_event.set()

# Wait for the thread to finish
thread.join()
print("Main thread done.")

In [ ]:
import threading
import time

def worker(name):
    import random

    seconds = random.randint(5, 20)
    print(f'{name} starting...sleeping for {seconds} seconds')
    time.sleep(seconds)
    print(f'{name} done')

# make a list of 10 threads/workers

threads = [threading.Thread(target=worker, args=(f'worker {num}',)) for num in range(1, 11)]
    
for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print('all threads finished')

## Let's use threads to get some real data

In [2]:
from threading import Thread
from urllib.request import urlopen
from urllib.parse import urlencode
import json
import time

API_KEY = '34d326cbd7a8e9b149ea30c5068d270d'

cities = [
    'Boulder', 'Atlanta', 'San Francisco', 'Reno', 'Rome',
    'Honolulu', 'Zurich', 'Dubai', 'Dublin', 'Hyderabad'
]

results = {}

def get_temperature(city):
    try:
        params = {
            'q': city,
            'units': 'imperial',
            'appid': API_KEY,
        }

        url = 'https://api.openweathermap.org/data/2.5/weather?' + urlencode(params)

        with urlopen(url, timeout=10) as resp:
            data = json.loads(resp.read().decode('utf-8'))

        if str(data.get('cod', 200)) != '200':
            raise RuntimeError(f"{data.get('cod')}: {data.get('message')}")

        results[city] = {
            'temperature': data['main']['temp'],
            'error': None
        }

    except Exception as e:
        results[city] = {
            'temperature': None,
            'error': str(e)
        }


threads = [
    Thread(target=get_temperature, args=(city,))
    for city in cities
]

start = time.time()

for t in threads:
    t.start()

for t in threads:
    t.join()

ok, failed = 0, 0

for city in cities:
    temperature = results[city]['temperature']
    error = results[city]['error']

    if error is None and temperature is not None:
        print(f'it is {temperature:.0f}°F in {city}')
        ok += 1
    else:
        print(f'[failed] {city}: {error}')
        failed += 1

print(f'Got {ok} temps ({failed} failed) in {time.time() - start:.2f} seconds')

it is 71°F in Boulder
it is 74°F in Atlanta
it is 63°F in San Francisco
it is 51°F in Reno
it is 69°F in Rome
it is 83°F in Honolulu
it is 41°F in Zurich
it is 82°F in Dubai
it is 64°F in Dublin
it is 80°F in Hyderabad
Got 10 temps (0 failed) in 0.14 seconds


### Shared Memory Model
* threads can all access the same global variables and shared objects
* that makes some designs simple, but it also means you must be careful when multiple threads read and write the same state

In [ ]:
import threading
import time

counter = 0

def increment():
    global counter
    
    for _ in range(100_000):
        # increment the counter, but do it in a roundabout way
        # so as to demonstrate what can happen when we share state
        newvar = counter
        newvar += 1
        time.sleep(.00001)
        counter = newvar

threads = [threading.Thread(target=increment) for _ in range(2)]

for t in threads:
    t.start()

for t in threads:
    t.join()

print(counter)

* the result may or may not be what you expect
* this is the kind of situation that introduces **race conditions**

## Race Conditions and Shared State
* a race condition happens when correctness depends on unpredictable timing between tasks
* when multiple threads update shared data, operations can overlap in unsafe ways
* a common tool for protecting shared state is a lock

In [ ]:
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100_000):
        with lock:
            newvar = counter
            newvar += 1
            time.sleep(.00001)
            counter = newvar

threads = [threading.Thread(target=increment) for _ in range(2)]

for t in threads:
    t.start()

for t in threads:
    t.join()

print(counter)

* locks reduce race conditions by allowing only one thread at a time into a critical section
* this improves correctness, but too much locking can also reduce concurrency and make code harder to reason about

## Multiprocessing
* the __`multiprocessing`__ module creates separate processes instead of threads
* processes have separate memory spaces, which makes them well suited for CPU-bound work
* a basic lifecycle looks similar to threading:
  * create a __`Process`__
  * call __`start()`__
  * later call __`join()`__


In [ ]:
# won't actually work in the notebook

import multiprocessing
import os

def worker():
    print(f'process id: {os.getpid()}')

if __name__ == '__main__':
    p = multiprocessing.Process(target=worker)
    p.start()
    p.join()

### The __`__name__ == '__main__'`__ Guard
* when using multiprocessing, process creation re-imports the main module
* multiprocessing examples should usually be protected with:
  * __`if __name__ == '__main__':`__
* without it, your code can rerun unexpectedly when child processes start

In [ ]:
%%writefile mp_helpers.py

# we'll put the worker code in a separate file and import it in order for 
# this to work in Jupyter (we could also put the whole thing in its own
# file and just run it from Jupyter)
def worker():
    print('child process running', flush=True)
    
    for num in range(1_000_000_000):
        if num and num % 100_000_000 == 0:
            print(f'{num:,}', flush=True)
        result = num * num
        
    print('child process finished')

In [ ]:
import multiprocessing
from mp_helpers import worker

if __name__ == '__main__':
    p = multiprocessing.Process(target=worker)
    p.start()
    p.join()

In [ ]:
%%writefile standalone.py
import multiprocessing
    
def worker():
    print('child process running', flush=True)
    
    for num in range(1_000_000_000):
        if num and num % 100_000_000 == 0:
            print(f'{num:,}', flush=True)
        result = num * num
        
    print('child process finished')
    
if __name__ == '__main__':
    p = multiprocessing.Process(target=worker)
    p.start()
    p.join()

In [ ]:
%run standalone.py

### Process Overhead Considerations
* processes can run in parallel, but they cost more than threads
* costs include:
  * process startup time
  * separate memory spaces
  * data serialization and transfer between processes
* multiprocessing is best when the tasks are large enough to justify the overhead

## Inter-Process Communication
* because processes do not share memory by default, they often communicate through message passing
* a common tool is __`multiprocessing.Queue`__

In [ ]:
%%writefile queue_ex.py
# let's create 3 child processes which run in parallel
import multiprocessing

def worker(q, worker_id):
    message = f'worker {worker_id} writes {worker_id * worker_id}'
    q.put(message)
    for num in range(1_000_000_000): # simulate doing some work for a while
        results = num * num


if __name__ == '__main__':
    q = multiprocessing.Queue()

    processes = [
        multiprocessing.Process(
            target=worker,
            args=(q, i)
        )
        for i in range(1, 4) # how many?
    ]

    for p in processes:
        p.start()

    results = [q.get() for _ in range(3)]

    for p in processes:
        p.join()

    print(results)

In [ ]:
%run queue_ex.py

### Message Passing vs. Shared State
* a useful mental model is:
  * threads often share state by default
  * processes usually communicate by passing messages
* message passing is often simpler and safer than sharing mutable state directly

## Sharing State Between Processes
* processes can share some state, but it requires explicit tools
* common options include:
  * __`Value`__ for a single shared value
  * __`Array`__ for a shared array
  * __`Manager`__ for shared proxy objects such as dicts and lists

In [ ]:
%%writefile shared_value.py
import multiprocessing

def increment(counter):
    for _ in range(10_000):
        counter.value += 1


if __name__ == '__main__':
    # creates a shared value that both child processes can access
    counter = multiprocessing.Value('i', 0)

    p1 = multiprocessing.Process(target=increment, args=(counter,))
    p2 = multiprocessing.Process(target=increment, args=(counter,))

    p1.start()
    p2.start()

    p1.join()
    p2.join()

    print(counter.value)

In [ ]:
%run shared_value.py

* this example can still show race-condition behavior because shared state is not automatically synchronized
* sharing state between processes often reintroduces the same kinds of coordination problems seen with threads

In [ ]:
%%writefile manager.py

# here we'll share an actual data stucture, a dict

import multiprocessing

def add_score(shared_dict, name, score):
    shared_dict[name] = score # this makes a  request to the manager process

if __name__ == '__main__':
    # start a special manager process
    with multiprocessing.Manager() as manager:
        # the below creates a proxy object that talks to the manager process
        scores = manager.dict()

        p1 = multiprocessing.Process(target=add_score, args=(scores, 'Alice', 95))
        p2 = multiprocessing.Process(target=add_score, args=(scores, 'Bob', 88))

        p1.start()
        p2.start()

        p1.join()
        p2.join()

        print(dict(scores))

In [ ]:
%run manager.py

* So are we all set? Not quite...
* the example above gives us:
  * shared access across processes
  * a dictionary-like interface
  * safer coordination than trying to share a normal dict directly
* so the manager enables shared state, but it does not remove the need to think about synchronization–a lock makes compound updates safe
  * if two processes update the managed object at the same time we have to worry about:
    * concurrent access
    * ordering effects
    * potential contention for the manager process

## Synchronization Between Processes
* multiprocessing also provides synchronization primitives such as:
  * __`Lock`__ (protects a critical section)
  * __`Semaphore`__ (limits how many workers can access a resource at once)

In [ ]:
%%writefile lock_ex.py
    
import multiprocessing

def increment(counter, lock):
    for _ in range(10_000):
        with lock:
            counter.value += 1

if __name__ == '__main__':
    counter = multiprocessing.Value('i', 0)
    lock = multiprocessing.Lock()

    p1 = multiprocessing.Process(target=increment, args=(counter, lock))
    p2 = multiprocessing.Process(target=increment, args=(counter, lock))

    p1.start()
    p2.start()

    p1.join()
    p2.join()

    print(counter.value)

In [ ]:
%run lock_ex.py

In [ ]:
%%writefile semaphore_ex.py
    
import multiprocessing
import time

def worker(sem, name):
    with sem:
        print(f'{time.time():.3f} | {name} entering', flush=True)
        time.sleep(1)
        print(f'{time.time():.3f} | {name} leaving', flush=True)

if __name__ == '__main__':
    sem = multiprocessing.Semaphore(2)
    processes = [
        multiprocessing.Process(target=worker, args=(sem, f'p{i}'))
        for i in range(4)
    ]

    for p in processes:
        p.start()

    for p in processes:
        p.join()

In [ ]:
%run semaphore_ex.py

## Common Pitfalls
* a few mistakes come up repeatedly:
  * using threads for CPU-bound work and expecting a speedup
    * (although with a GIL-free version of Python we should see speedup)
  * forgetting to call __`join()`__
  * mutating shared state without synchronization
  * assuming processes share normal Python variables
  * forgetting the __`__main__`__ guard with multiprocessing
  * using multiprocessing for tasks too small to justify process overhead

A good habit is to first ask:

> what is the bottleneck, and what kind of coordination does this design require?


## Higher-Level Concurrency with __`concurrent.futures`__
* the low-level modules are important for understanding the model, but many real programs use higher-level tools
* __`concurrent.futures`__ module provides:
  * __`ThreadPoolExecutor`__ for I/O-bound work
  * __`ProcessPoolExecutor`__ for CPU-bound work
* these abstractions are often easier to read and maintain than manually managing threads and processes

## Let's get those 10 temperatures again, but this time with a __`ThreadPoolExecutor`__

In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.request import urlopen
from urllib.parse import urlencode
import json
import time

API_KEY = '34d326cbd7a8e9b149ea30c5068d270d'

cities = [
    'Boulder', 'Atlanta', 'San Francisco', 'Reno', 'Rome',
    'Honolulu', 'Zurich', 'Dubai', 'Dublin', 'Hyderabad'
]

def get_temperature(city):
    try:
        params = {
            'q': city,
            'units': 'imperial',
            'appid': API_KEY,
        }
        url = 'https://api.openweathermap.org/data/2.5/weather?' + urlencode(params)

        with urlopen(url, timeout=10) as resp:
            data = json.loads(resp.read().decode('utf-8'))

        if str(data.get('cod', 200)) != '200':
            raise RuntimeError(f"{data.get('cod')}: {data.get('message')}")

        return city, data['main']['temp'], None

    except Exception as e:
        return city, None, str(e)


start = time.time()
ok, failed = 0, 0

with ThreadPoolExecutor(max_workers=len(cities)) as executor:
    # each submit we do creates a "future"–
    # a placeholder for a result that hasn't finished yet
    futures = [executor.submit(get_temperature, city) for city in cities]

    # give me each "future" as soon as it finishes–not necessarily in the
    # original order, but in the order of completion...
    for future in as_completed(futures):
        city, temperature, error = future.result()

        if error is None and temperature is not None:
            print(f'it is {temperature:.0f}°F in {city}')
            ok += 1
        else:
            print(f'[failed] {city}: {error}')
            failed += 1

print(f'Got {ok} temps ({failed} failed) in {time.time() - start:.2f} seconds')

it is 71°F in Boulder
it is 83°F in Honolulu
it is 80°F in Hyderabad
it is 63°F in San Francisco
it is 82°F in Dubai
it is 41°F in Zurich
it is 74°F in Atlanta
it is 69°F in Rome
it is 51°F in Reno
it is 64°F in Dublin
Got 10 temps (0 failed) in 0.14 seconds


### __`ProcessPoolExecutor`__ for CPU-Bound Work


In [ ]:
%%writefile processpool_ex.py
from concurrent.futures import ProcessPoolExecutor

def cpu_heavy_task(n):
    total = 0

    for i in range(300_000_000):
        total += (i * n) % 7

    return f'input={n}, result={total}'


if __name__ == '__main__':
    numbers = [1, 2, 3, 4, 5]

    with ProcessPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(cpu_heavy_task, numbers))

    for result in results:
        print(result)

In [ ]:
%run processpool_ex.py

* these executors are often the best practical starting point once you understand the lower-level model


## __`asyncio`__

* Python's built-in framework for asynchronous I/O concurrency
* designed for workloads that spend lots of time waiting on I/O, but it does so without requiring multiple threads
* _coroutines_: program components that allow execution to be suspended and resumed, generalizing subroutines for cooperative multitasking
* _event loop_: a programming construct that waits for and dispatches events or messages in a program (gives us asynchronous, non-blocking I/O operations despite using a single thread)
  * continuously monitors an event queue, executing callbacks when tasks complete
      * callback
      * is a function you give to the system to run later when an event occurs, e.g,. "When the download finishes, call this function"
* well-suited for implementing familiar program components such as cooperative tasks, exceptions, event loops, iterators, infinite lists and pipes

* what are events?
  * a timer finishing
  * a network response arriving
  * user input
  * a button click
  * a file becoming ready
  * a socket receiving data

* key ideas:
  * event loop
  * __`async`__ / __`await`__
  * cooperative multitasking

In [10]:
import asyncio

async def task(name, delay):
    for i in range(3):
        print(f'{name}: step {i + 1}')

        # pause here and let other tasks run
        # this is cooperative multitasking
        await asyncio.sleep(delay)

    print(f'{name}: done')


async def main():
    # start both tasks together
    await asyncio.gather(
        task('A', 1),
        task('B', 1.5)
    )

# in a normal Python script
# asyncio.run(main())

# in Jupyter:
await main()

A: step 1
B: step 1
A: step 2
B: step 2
A: step 3
B: step 3
A: done
B: done


## Let's try the temperature getting one more time...
* when we did it with threads, we had 10 of them, each of which could block, waiting for the result
* but with async / await we have
  * one worker
  * many paused tasks
  * switch at __`await`__ points

In [7]:
import time
import asyncio
import aiohttp # need an async version of http module
# it's probably not installed...
# !pip install aiohttp

API_KEY = '34d326cbd7a8e9b149ea30c5068d270d'

cities = [
    'Boulder', 'Atlanta', 'San Francisco', 'Reno', 'Rome',
    'Honolulu', 'Zurich', 'Dubai', 'Dublin', 'Hyderabad'
]

# async function = coroutine
# this can pause with await and let other work run
async def get_temperature(session, city):
    try:
        params = {
            'q': city,
            'units': 'imperial',
            'appid': API_KEY,
        }

        url = 'https://api.openweathermap.org/data/2.5/weather'

        # async HTTP request
        # while waiting for the network response,
        # the event loop can run other city requests
        async with session.get(url, params=params, timeout=10) as resp:
            # await = "pause here until JSON is ready"
            # during this pause, other tasks continue running
            data = await resp.json()

        if str(data.get('cod', 200)) != '200':
            raise RuntimeError(f"{data.get('cod')}: {data.get('message')}")

        return city, data['main']['temp'], None

    except Exception as e:
        return city, None, str(e)


# main async coordinator
async def main():
    start = time.time()
    ok, failed = 0, 0

    # one shared HTTP session for all requests
    async with aiohttp.ClientSession() as session:
        # create all tasks immediately
        # this does NOT run them yet
        # it creates coroutine objects
        tasks = [get_temperature(session, city) for city in cities]

        # asyncio.gather() schedules them all together
        # all requests can overlap while waiting on I/O
        # this is concurrency without creating threads
        results = await asyncio.gather(*tasks)

    # once all requests are complete,
    # process the returned results
    for city, temperature, error in results:
        if error is None and temperature is not None:
            print(f'it is {temperature:.0f}°F in {city}')
            ok += 1
        else:
            print(f'[failed] {city}: {error}')
            failed += 1

    print(f'Got {ok} temps ({failed} failed) in {time.time() - start:.2f} seconds')

# in Jupyter:
# await main()

# in a normal Python script:
# if __name__ == '__main__':
#     asyncio.run(main())

await main()

it is 65°F in Boulder
it is 72°F in Atlanta
it is 63°F in San Francisco
it is 51°F in Reno
it is 67°F in Rome
it is 83°F in Honolulu
it is 41°F in Zurich
it is 82°F in Dubai
it is 63°F in Dublin
it is 82°F in Hyderabad
Got 10 temps (0 failed) in 0.21 seconds


### Event Loop and __`async`__ / __`await`__
* the event loop coordinates tasks and resumes them when they are ready to continue
* with __`await`__, a coroutine can pause at an I/O boundary and let other work run
* that is why __`asyncio`__ is often described as **cooperative multitasking**
* tasks are not preempted by the operating system in the same way threads are
* instead, they cooperate by yielding control at __`await`__ points

### When __`asyncio`__ Fits Well

__`asyncio`__ is a strong fit when:

* work is I/O-bound
* you have many concurrent tasks
* the libraries you are using support async APIs

It is usually **not** the right tool for CPU-bound work


## Choosing the Right Tool

A simple decision guide looks like this:

* I/O-bound and simple
  * __`threading`__ or __`ThreadPoolExecutor`__
* I/O-bound and many concurrent tasks
  * __`asyncio`__
* CPU-bound
  * __`multiprocessing`__ or __`ProcessPoolExecutor`__
* shared mutable state
  * be careful
  * prefer message passing when possible

Another good rule of thumb:

> start with the simplest tool that matches the bottleneck


## Summary

Key ideas from this notebook:

* concurrency and parallelism are related, but different
* I/O-bound and CPU-bound workloads need different tools
* the GIL limits CPU-bound threading in CPython
* threads share memory and are useful for waiting
* processes have separate memory and are useful for computation
* queues and message passing are often simpler than shared state
* locks and semaphores help coordinate access to shared resources
* __`concurrent.futures`__ provides practical higher-level abstractions
* __`asyncio`__ is a modern tool for I/O-heavy concurrency without threads
* the best tool depends on the bottleneck and the coordination model

The goal is not to use every concurrency tool

The goal is to choose the one that fits the problem most naturally
